# Laser Off Code

In [ ]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

# Import

In [6]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
initialise_or_create_database_at("./2026-03-10_SNSPD4.db")
from functions import osc_set_standard, osc_check_standard, capture_trace, capture_trace_simple
import snspd 
params = snspd.snspd()

# Set up experiment
exp_name = 'SNSPD4_23_03_2026'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Experiment loaded. Last ID no: 488


# Instruments

In [19]:
station = Station(config_file="friesland.yaml")
dmm = station.load_instrument("dmm", revive_instance=True)
yoko = station.load_instrument("yoko", revive_instance=True)
laser = station.load_instrument("laser", revive_instance=True)
MS = station.load_instrument("mso5", revive_instance=True)
pm100usb = station.load_instrument("pm100usb", revive_instance=True)
pms120 = station.load_instrument("pms120", revive_instance=True)
tc = station.load_instrument("fridge", revive_instance=True)
p_att = station.load_instrument("dmm_keithley", revive_instance=True) # excluding from snapshot because none of the parameters work anyway

# Trace Captures

Old Device 

Set trigger at a lower level, base on set thresholds 

Parameters from SNSPD4_5_Counts_vs_Current.ipynb
v_attenuator = 4.7
v_scale_old_device = 50e-3
current_range = np.arange(1e-7, 4.1e-6, 1e-7)
device_name = 'Line 2 Old Device'

In [15]:
from functions import set_thresholds
# Parameters from SNSPD4_1_Trace_Capture.ipynb
h_time = 3e-6 # time for trace acquisition
h_pos = 10
v_scale=25e-3
v_trigger = -25e-3

osc_set_standard(MS, v_trigger = v_trigger, v_scale=v_scale, h_time=h_time, h_pos=h_pos)

# trigger looks good 

In [12]:
############################ TURN LASER ON ############################ 
laser.enable(True)

time.sleep(10)
print(f'Laser enable status: {laser.enable()}')


Laser enable status: True


In [13]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: False


Sweeping current: dark conditions fibre connected and shielded 

In [ ]:
# Parameters from SNSPD4_1_Trace_Capture.ipynb
h_time = 3e-6 # time for trace acquisition
h_pos = 10
v_scale=25e-3
v_trigger = -25e-3

print(v_scale) 
print(h_time)
print(h_pos)
print(v_trigger)

osc_set_standard(MS, v_trigger = v_trigger, v_scale=v_scale, h_time=h_time, h_pos=h_pos)


if MS.channels[0].clipping(): 
    print('Error: Clipping')

# Set to 4.7 for 100dB attenuation
p_att.write('VOLT 4.7')
time.sleep(5)

currents = np.arange(0.9e-6, 4.1e-6, 1e-7) # Same current range as ID 483, but start just before 1e-6 because that is the first result where there are counts

# Set to starting current 
yoko.current(currents[0])


for curr in currents:

    # Set current 
    yoko.current(curr)

    print(f'Current is {yoko.current()}')

    # capture the trace 
    capture_trace(MS=MS, dmm=dmm, yoko=yoko, p_att=p_att, station=station)

    time.sleep(5)
    
    

0.025
3e-06
10
-0.025
Current is 9e-07
update station
Starting experimental run with id: 489. 
489
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 6.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;955.000E-12;188;"V";1.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1
Current is 1e-06
update station
Starting experimental run with id: 490. 
490
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 6.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;955.000E-12;188;"V";1.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1
Current is 1.1e-06
update station
Starting experimental run with id: 491. 
491
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 6.4V/div, 300ns/div, 1875 points, Sample mode";1875;Y;LINEAR;"s";1.60E-9;955.000E-12;188;"V";1.0000E-3;0.0E+0;0.0E+0;TIME;ANALOG;0.0E+0;0.0E+0;1
Current is 1.2e-06
update station
Starting experimental run with id: 492. 
492
1;8;BINARY;RI;INTEGER;MSB;"Ch1, DC coupling, 6.4V/div, 300ns/div, 1875 points, Sample mode";1875

Sweeping current: fibre uncapped 

In [ ]:
# Parameters from SNSPD4_1_Trace_Capture.ipynb
h_time = 3e-6 # time for trace acquisition
h_pos = 10
v_scale=25e-3
v_trigger = -25e-3

print(v_scale) 
print(h_time)
print(h_pos)
print(v_trigger)

osc_set_standard(MS, v_trigger = v_trigger, v_scale=v_scale, h_time=h_time, h_pos=h_pos)


if MS.channels[0].clipping(): 
    print('Error: Clipping')

# Set to 4.7 for 100dB attenuation
p_att.write('VOLT 4.7')
time.sleep(5)

currents = np.arange(0.9e-6, 4.1e-6, 1e-7) # Same current range as ID 483, but start just before 1e-6 because that is the first result where there are counts

# Set to starting current 
yoko.current(currents[0])


for curr in currents:

    # Set current 
    yoko.current(curr)

    print(f'Current is {yoko.current()}')

    # capture the trace 
    capture_trace(MS=MS, dmm=dmm, yoko=yoko, p_att=p_att, station=station)

    time.sleep(5)
    
    

Sweeping current: laser on 

In [ ]:
# Parameters from SNSPD4_1_Trace_Capture.ipynb
h_time = 3e-6 # time for trace acquisition
h_pos = 10
v_scale=25e-3
v_trigger = -25e-3

print(v_scale) 
print(h_time)
print(h_pos)
print(v_trigger)

osc_set_standard(MS, v_trigger = v_trigger, v_scale=v_scale, h_time=h_time, h_pos=h_pos)


if MS.channels[0].clipping(): 
    print('Error: Clipping')

# Set to 4.7 for 100dB attenuation
p_att.write('VOLT 4.7')
time.sleep(5)

currents = np.arange(0.9e-6, 4.1e-6, 1e-7) # Same current range as ID 483, but start just before 1e-6 because that is the first result where there are counts

# Set to starting current 
yoko.current(currents[0])


############################ TURN LASER ON ############################ 
laser.enable(True)

time.sleep(10)
print(f'Laser enable status: {laser.enable()}')

for curr in currents:

    # Set current 
    yoko.current(curr)

    print(f'Current is {yoko.current()}')

    # capture the trace 
    capture_trace(MS=MS, dmm=dmm, yoko=yoko, p_att=p_att, station=station)

    time.sleep(5)

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')
    

Sweeping attenuation

In [11]:
# Parameters from SNSPD4_1_Trace_Capture.ipynb
h_time = 3e-6 # time for trace acquisition
h_pos = 10
v_scale=25e-3
v_trigger = -25e-3

print(v_scale) 
print(h_time)
print(h_pos)
print(v_trigger)

osc_set_standard(MS, v_trigger = v_trigger, v_scale=v_scale, h_time=h_time, h_pos=h_pos)

yoko.current(2.1e-6)


if MS.channels[0].clipping(): 
    print('Error: Clipping')


############################ TURN LASER ON ############################ 
laser.enable(True)

time.sleep(10)
print(f'Laser enable status: {laser.enable()}')


for v in np.arange(3.4, 6.1, 0.1)[::-1]:

    # Set attenuator voltage 
    p_att.write(f'VOLT {v}')

    time.sleep(20) # to let the trace change because of that weird thing that happens with the trace not adjusting immediately

    print(f'Voltage on attenuator is {p_att.ask('VOLT?')}')

    for i in range(2):

        # capture the trace 
        capture_trace(MS, dmm, p_att, station=station)

        time.sleep(5)
    
    
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

SyntaxError: unmatched ')' (2227862928.py, line 12)